
# 🧪 Fairness Lab — Measuring & Mitigating Bias (Adversarial Debiasing) — **Corrected version**

> **Format**: guided lab session (≈ 1h30) with short exercises + immediate verification  
> **Goal**: implement **fairness metrics** and an **adversarial debiasing** strategy.

---

## 🎯 Learning objectives
By the end of this lab, you will be able to:
1. Train a simple **MLP classifier** on top of fixed BERT embeddings
2. Implement and report fairness metrics:
   - **Disparate Impact (DI)**
   - **Equalized Opportunity gap (EO gap)**
3. Train an **attacker (adversary)** that predicts a sensitive attribute (gender) from the classifier output
4. Train a **defended classifier** that preserves task accuracy while reducing leakage about the sensitive attribute

---

## ⏱️ Suggested timing
- **Part 0 — Setup & data** (10 min)
- **Part 1 — Implement metrics + baseline MLP** (20 min)
- **Part 2 — Train job classifier (baseline)** (20 min)
- **Part 3 — Train attacker** (15 min)
- **Part 4 — Adversarial defense training** (25 min)
- **Bonus** (if time)

---

## 🧾 Dataset (expected)
A set of short biographies, each with:
- `classe` (profession label): 0 = nurse, 1 = doctor (binary)
- `gender` (sensitive attribute): 0/1
- `bert` (768-dim embedding)

If the provided `data.pk` is not found, this notebook will generate a small **synthetic** dataset
so that all code remains runnable.


## 0) Setup

In [ ]:

# If you get an ImportError, install dependencies (uncomment if needed):
# !pip -q install torch numpy scikit-learn tqdm

import os
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tqdm import tqdm

torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("PyTorch:", torch.__version__)



## 0.b) Load data

We expect a `data.pk` file containing a pandas-like object where:
- `data["bert"]` is a list/array of 768-dim vectors
- `data["classe"]` is a binary label
- `data["gender"]` is a binary sensitive attribute

The original TD used `pickle.load(open("data.pk",'rb'))`.
We keep the same idea but add a safe fallback.


In [ ]:

DATA_PATH = "../data/data.pk"  # expected next to the notebook (or in current working directory)

def make_synthetic_dataset(n=4000, dim=768, p_gender=0.5, bias_strength=1.0):
    """
    Synthetic binary classification with a spurious correlation to gender.
    - y depends on a 'true' signal + a bias term correlated with gender.
    """
    A = (np.random.rand(n) < p_gender).astype(int)  # gender (0/1)
    z = np.random.randn(n, dim).astype(np.float32)  # embeddings
    w_true = np.random.randn(dim).astype(np.float32) / np.sqrt(dim)

    logits = z @ w_true
    logits = logits + bias_strength * (A - 0.5)  # bias: y more likely to be 1 for A=1
    y = (logits > np.median(logits)).astype(int)
    return {"bert": z, "classe": y, "gender": A}

if os.path.exists(DATA_PATH):
    data = pickle.load(open(DATA_PATH, "rb"))
    print(f"Loaded {DATA_PATH}.")
else:
    print(f"⚠️ {DATA_PATH} not found. Using a synthetic dataset (for runnable notebook).")
    data = make_synthetic_dataset(n=3000, dim=768, bias_strength=1.5)

def get_col(obj, key):
    if isinstance(obj, dict):
        return obj[key]
    return obj[key].values

X = np.vstack(get_col(data, "bert")).astype(np.float32)
y = np.asarray(get_col(data, "classe"), dtype=int).reshape(-1)
A = np.asarray(get_col(data, "gender"), dtype=int).reshape(-1)

print("X:", X.shape, "y:", y.shape, "A:", A.shape)
print("Unique y:", np.unique(y), "| Unique A:", np.unique(A))


## 0.c) Train/test split + DataLoaders

In [ ]:

batch_size = 128

X_train, X_test, y_train, y_test, A_train, A_test = train_test_split(
    X, y, A, test_size=0.2, random_state=42, stratify=y
)

x_train = torch.tensor(X_train)
y_train_t = torch.tensor(y_train, dtype=torch.long)
a_train_t = torch.tensor(A_train, dtype=torch.long)

x_test = torch.tensor(X_test)
y_test_t = torch.tensor(y_test, dtype=torch.long)
a_test_t = torch.tensor(A_test, dtype=torch.long)

train_ds = TensorDataset(x_train, y_train_t, a_train_t)
test_ds  = TensorDataset(x_test,  y_test_t,  a_test_t)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=512, shuffle=False)

print("Train:", len(train_ds), "| Test:", len(test_ds))



# 1) Fairness metrics + baseline MLP

We will compute and report:
- **Accuracy**
- **Disparate Impact (DI)**  
  \[ DI = P(\hat{Y}=1 \mid A=0) / P(\hat{Y}=1 \mid A=1) \]  
  (You may swap which group is in numerator/denominator depending on the *privileged* group. Here we use **A=1** as reference.)
- **Equalized Opportunity gap (EO gap)**  
  \[ EO\_gap = |TPR_{A=0} - TPR_{A=1}| \]  
  where \(TPR = P(\hat{Y}=1 \mid Y=1, A=a)\).

> ⚠️ When a denominator is 0, we add a small epsilon for numerical safety.



## Exercise 1 — Implement the evaluation function (DI + EO gap)

Complete the function skeleton (then compare with the correction cell).

### ✍️ To complete


In [ ]:

def compute_di(y_pred, A, positive_label=1, ref_group=1, eps=1e-12):
    """
    Disparate Impact: P(yhat=1|A!=ref) / P(yhat=1|A=ref)
    For binary A in {0,1}, 'ref_group=1' means denominator uses A=1.
    """
    # TODO
    pass

def compute_eo_gap(y_true, y_pred, A, positive_label=1, eps=1e-12):
    """
    Equalized Opportunity gap: |TPR(A=0) - TPR(A=1)|
    """
    # TODO
    pass

@torch.no_grad()
def evaluation(model, dataloader):
    """Returns: accuracy, DI, EO_gap"""
    model.eval()
    Y_all, yhat_all, A_all = [], [], []
    for xb, yb, ab in dataloader:
        xb = xb.to(device)
        logits = model(xb)
        yhat = torch.argmax(logits, dim=1).cpu().numpy()
        Y_all.append(yb.numpy())
        yhat_all.append(yhat)
        A_all.append(ab.numpy())

    Y_all = np.concatenate(Y_all)
    yhat_all = np.concatenate(yhat_all)
    A_all = np.concatenate(A_all)

    # TODO: compute metrics here
    # acc = ...
    # di = ...
    # eo = ...
    return acc, di, eo


### ✅ Correction

In [ ]:

def compute_di(y_pred, A, positive_label=1, ref_group=1, eps=1e-12):
    A = np.asarray(A).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    mask_ref = (A == ref_group)
    mask_other = ~mask_ref

    p_ref = np.mean(y_pred[mask_ref] == positive_label) if mask_ref.any() else 0.0
    p_other = np.mean(y_pred[mask_other] == positive_label) if mask_other.any() else 0.0

    return (p_other + eps) / (p_ref + eps)

def compute_eo_gap(y_true, y_pred, A, positive_label=1, eps=1e-12):
    A = np.asarray(A).reshape(-1)
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    m0 = (A == 0) & (y_true == positive_label)
    tpr0 = np.mean(y_pred[m0] == positive_label) if m0.any() else 0.0

    m1 = (A == 1) & (y_true == positive_label)
    tpr1 = np.mean(y_pred[m1] == positive_label) if m1.any() else 0.0

    return abs(tpr0 - tpr1)

@torch.no_grad()
def evaluation(model, dataloader):
    model.eval()
    Y_all, yhat_all, A_all = [], [], []
    for xb, yb, ab in dataloader:
        xb = xb.to(device)
        logits = model(xb)
        yhat = torch.argmax(logits, dim=1).cpu().numpy()
        Y_all.append(yb.numpy())
        yhat_all.append(yhat)
        A_all.append(ab.numpy())

    Y_all = np.concatenate(Y_all)
    yhat_all = np.concatenate(yhat_all)
    A_all = np.concatenate(A_all)

    acc = accuracy_score(Y_all, yhat_all)
    di = compute_di(yhat_all, A_all, positive_label=1, ref_group=1)
    eo = compute_eo_gap(Y_all, yhat_all, A_all, positive_label=1)
    return acc, di, eo



## Exercise 2 — Implement a simple MLP

We use a **simple MLP**:
- Linear: `init_dim → 512`
- ReLU
- BatchNorm
- Linear: `512 → out_dim`

We will instantiate:
- `classifier = MLP(init_dim=768, out_dim=2)` (profession)
- `attacker  = MLP(init_dim=2,   out_dim=2)` (gender from classifier logits)

### ✍️ To complete


In [ ]:

class MLP(nn.Module):
    def __init__(self, init_dim, hidden_dim=512, out_dim=2):
        super().__init__()
        # TODO
        pass

    def forward(self, x):
        # TODO
        pass

classifier = MLP(init_dim=768, out_dim=2).to(device)
attacker   = MLP(init_dim=2, out_dim=2).to(device)

print(classifier)
print(attacker)


### ✅ Correction

In [ ]:

class MLP(nn.Module):
    def __init__(self, init_dim, hidden_dim=512, out_dim=2):
        super().__init__()
        self.fc1 = nn.Linear(init_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, out_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = F.relu(x)
        x = self.bn1(x)
        x = self.fc2(x)
        return x

classifier = MLP(init_dim=768, out_dim=2).to(device)
attacker   = MLP(init_dim=2, out_dim=2).to(device)

print(classifier)
print(attacker)


### Baseline evaluation (random init)

In [ ]:

acc, di, eo = evaluation(classifier, test_loader)
print(f"Initial (random) | acc={acc:.3f} | DI={di:.3f} | EO_gap={eo:.3f}")



# 2) Train a baseline job classifier

**Task**: predict profession `y` from BERT embeddings `x`.

Report:
- Accuracy
- DI
- EO gap


In [ ]:

epochs = 10
lr = 5e-4
criterion = nn.CrossEntropyLoss()

classifier = MLP(init_dim=768, out_dim=2).to(device)
opt_clf = torch.optim.Adam(classifier.parameters(), lr=lr)

for epoch in range(1, epochs + 1):
    classifier.train()
    running = 0.0
    for xb, yb, ab in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        opt_clf.zero_grad()
        logits = classifier(xb)
        loss = criterion(logits, yb)
        loss.backward()
        opt_clf.step()

        running += loss.item() * xb.size(0)

    train_loss = running / len(train_loader.dataset)
    acc, di, eo = evaluation(classifier, test_loader)
    if epoch in [1, 3, 5, 10]:
        print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | acc={acc:.3f} | DI={di:.3f} | EO_gap={eo:.3f}")



# 3) Train an attacker (gender predictor) from classifier outputs

We now train an **attacker** that predicts `A` from the classifier output logits:
- Input to attacker: `z = classifier(x)` (shape `(batch, 2)`)
- Target: `A`

**Interpretation**:
- High attacker accuracy ⇒ the classifier output carries information about gender (leakage).


In [ ]:

attacker = MLP(init_dim=2, out_dim=2).to(device)
opt_att = torch.optim.Adam(attacker.parameters(), lr=1e-3)
criterion_att = nn.CrossEntropyLoss()

# Freeze classifier for attacker training
classifier.eval()

for epoch in range(1, 11):
    attacker.train()
    running = 0.0
    y_true_all, y_pred_all = [], []

    for xb, yb, ab in train_loader:
        xb = xb.to(device)
        ab = ab.to(device)

        with torch.no_grad():
            z = classifier(xb)  # (batch, 2)

        opt_att.zero_grad()
        a_logits = attacker(z)
        loss = criterion_att(a_logits, ab)
        loss.backward()
        opt_att.step()

        running += loss.item() * xb.size(0)

        y_pred_all.append(torch.argmax(a_logits, dim=1).detach().cpu().numpy())
        y_true_all.append(ab.detach().cpu().numpy())

    att_acc = accuracy_score(np.concatenate(y_true_all), np.concatenate(y_pred_all))
    if epoch in [1, 3, 5, 10]:
        print(f"Attacker epoch {epoch:02d} | loss={running/len(train_loader.dataset):.4f} | acc={att_acc:.3f}")



# 4) Adversarial debiasing (defense training)

We re-train a new classifier with an adversarial objective:

1. **Update attacker** to minimize  
   `L_att(attacker(classifier(x).detach()), A)`
2. **Update classifier** to minimize  
   `L_task(classifier(x), y) - λ * L_att(attacker(classifier(x)), A)`

Subtracting the attacker loss means the classifier is trained to **maximize** it (confuse the attacker).

Try different `lambda_adv` values to observe the tradeoff.


In [ ]:

epochs = 10
lr_clf = 5e-4
lr_att = 1e-3
lambda_adv = 0.5  # try 0.1, 0.5, 1.0

clf_def = MLP(init_dim=768, out_dim=2).to(device)
att_def = MLP(init_dim=2, out_dim=2).to(device)

opt_clf = torch.optim.Adam(clf_def.parameters(), lr=lr_clf)
opt_att = torch.optim.Adam(att_def.parameters(), lr=lr_att)

crit_task = nn.CrossEntropyLoss()
crit_att  = nn.CrossEntropyLoss()

for epoch in range(1, epochs + 1):
    clf_def.train()
    att_def.train()

    running_task, running_att = 0.0, 0.0

    for xb, yb, ab in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        ab = ab.to(device)

        # (1) attacker step
        opt_att.zero_grad()
        with torch.no_grad():
            z_det = clf_def(xb).detach()
        a_logits = att_def(z_det)
        loss_att = crit_att(a_logits, ab)
        loss_att.backward()
        opt_att.step()

        # (2) classifier step
        opt_clf.zero_grad()
        z = clf_def(xb)
        loss_task = crit_task(z, yb)

        a_logits_for_clf = att_def(z)
        loss_att_for_clf = crit_att(a_logits_for_clf, ab)

        loss_clf = loss_task - lambda_adv * loss_att_for_clf
        loss_clf.backward()
        opt_clf.step()

        running_task += loss_task.item() * xb.size(0)
        running_att  += loss_att.item() * xb.size(0)

    acc, di, eo = evaluation(clf_def, test_loader)
    if epoch in [1, 3, 5, 10]:
        print(
            f"Epoch {epoch:02d} | "
            f"task_loss={running_task/len(train_loader.dataset):.4f} | "
            f"att_loss={running_att/len(train_loader.dataset):.4f} | "
            f"acc={acc:.3f} | DI={di:.3f} | EO_gap={eo:.3f}"
        )



## Attacker leakage after defense

Train a fresh attacker on top of the defended classifier outputs.  
Lower attacker accuracy usually indicates **less gender information** in classifier outputs.


In [ ]:

att_post = MLP(init_dim=2, out_dim=2).to(device)
opt_post = torch.optim.Adam(att_post.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()

clf_def.eval()
for epoch in range(1, 6):
    att_post.train()
    y_true_all, y_pred_all = [], []

    for xb, yb, ab in train_loader:
        xb = xb.to(device)
        ab = ab.to(device)

        with torch.no_grad():
            z = clf_def(xb)

        opt_post.zero_grad()
        a_logits = att_post(z)
        loss = crit(a_logits, ab)
        loss.backward()
        opt_post.step()

        y_true_all.append(ab.detach().cpu().numpy())
        y_pred_all.append(torch.argmax(a_logits, dim=1).detach().cpu().numpy())

    acc_att = accuracy_score(np.concatenate(y_true_all), np.concatenate(y_pred_all))
    if epoch in [1, 3, 5]:
        print(f"Post-defense attacker epoch {epoch:02d} | acc={acc_att:.3f}")



# ⭐ Bonus (if time)

## Bonus A — Compute metrics for both labels
Sometimes fairness is discussed for different “positive outcomes”.  
Compute DI/EO for `positive_label=0` too and compare.

## Bonus B — Choose the DI reference group
DI depends on the denominator group. Try `ref_group=0` and interpret.

## Bonus C — Add Demographic Parity gap
\[ DP\_gap = |P(\hat{Y}=1|A=0) - P(\hat{Y}=1|A=1)| \]

## Bonus D — Multiple adversaries (reading)
Han et al. (EACL 2021) suggest that ensembles of adversaries can improve fairness.
Adapt the defense to include multiple attackers trained in parallel.


In [ ]:

# Bonus C: Demographic Parity gap (example implementation)
def demographic_parity_gap(y_pred, A, positive_label=1):
    A = np.asarray(A).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    p0 = np.mean(y_pred[A == 0] == positive_label) if np.any(A == 0) else 0.0
    p1 = np.mean(y_pred[A == 1] == positive_label) if np.any(A == 1) else 0.0
    return abs(p0 - p1)

@torch.no_grad()
def evaluation_plus(model, dataloader):
    acc, di, eo = evaluation(model, dataloader)

    model.eval()
    Y_all, yhat_all, A_all = [], [], []
    for xb, yb, ab in dataloader:
        xb = xb.to(device)
        yhat = torch.argmax(model(xb), dim=1).cpu().numpy()
        Y_all.append(yb.numpy())
        yhat_all.append(yhat)
        A_all.append(ab.numpy())

    yhat_all = np.concatenate(yhat_all)
    A_all = np.concatenate(A_all)

    dp = demographic_parity_gap(yhat_all, A_all)
    return acc, di, eo, dp

acc, di, eo, dp = evaluation_plus(clf_def, test_loader)
print(f"Defended | acc={acc:.3f} | DI={di:.3f} | EO_gap={eo:.3f} | DP_gap={dp:.3f}")



---
## ✅ End of Lab

**What to report** (baseline vs defended):
- Accuracy
- DI (closer to 1 is “better” for this ratio form)
- EO gap (closer to 0 is better)
- Attacker accuracy (lower suggests less leakage)

**Discussion prompts**:
- What happens when you increase `lambda_adv`?
- Is there an accuracy–fairness tradeoff?
- Which metric is appropriate for your application?
